# 04 · Regression code overview

[`02_linear_regression`](02_linear_regression.ipynb) and
[`03_logistic_regression`](03_logistic_regression.ipynb) covered the *what*:
call `.fit(data)`, pick an optimizer, read off `weights` and `bias`. This
notebook covers the *how* — the shared implementation behind both models, in
`dolcestat.linear_models`.

Both regressions are built from the same abstract base, `GLMAbstract`, and
both can be fit up to three interchangeable ways:

| strategy | mechanism | gradients |
|---|---|---|
| gradient descent | the shared `Trainer` loop → `optimizer.step` | backprop |
| newton | its own loop, one joint Hessian on `[X \| 1]` | hand-derived |
| closed form | one matrix solve, `(XᵀX)⁻¹Xᵀy` | none |

We'll walk through each in turn, reading the source alongside small
experiments that check it does what it claims.

## A GLM *is* a one-layer network

`GLMAbstract` doesn't reimplement linear algebra: a generalized linear
model's prediction is `activation(XW + b)`, which is exactly what a
`DenseLayer` computes. So each model **owns** a one-layer `Sequential` and
only fills in the pieces that make it linear vs. logistic regression — the
activation, the loss, and (for the iterative strategies) how gradients and
curvature are computed.

In [1]:
from dolcestat.linear_models import LinearRegression, LogisticRegression

spec = ["_loss_class", "_activation_class", "_analyzer_class", "_allowed_optimizers"]
for model_cls in (LinearRegression, LogisticRegression):
    print(model_cls.__name__)
    for attr in spec:
        print(f"  {attr:<18} {getattr(model_cls, attr)}")

LinearRegression
  _loss_class        <class 'dolcestat.core.losses.MeanSquaredError'>
  _activation_class  <class 'dolcestat.core.activations.LU'>
  _analyzer_class    <class 'dolcestat.metrics.analyzer.LinearRegressionAnalyzer'>
  _allowed_optimizers {'closed form', 'newton', 'gradient descent'}
LogisticRegression
  _loss_class        <class 'dolcestat.core.losses.BinaryCrossEntropy'>
  _activation_class  <class 'dolcestat.core.activations.Sigmoid'>
  _analyzer_class    <class 'dolcestat.metrics.analyzer.ClassificationAnalyzer'>
  _allowed_optimizers {'newton', 'gradient descent'}


## Building the network — `_build_model`

`fit(data)` first calls `_build_model(n_features)`, identical for both
models:

```python
self.model = Sequential(
    DenseLayer(
        activation=self._activation_class(),
        input_size=n_features,
        output_size=1,
        parameters_initializer=ZeroParametersInitializer(),
    )
)
```

`fit` then dispatches on the optimizer: `"closed form"` and `"newton"` go
through `_fit_direct` (one shot, works on the bias-augmented matrix
`[X | 1]` directly); `"gradient descent"` goes through `_fit_iterative`,
which hands the network to the shared `Trainer`. Either way, the run ends by
writing into the same `DenseLayer.W` / `.b`, so `predict` doesn't care which
one ran.

```mermaid
flowchart TD
    A["LinearRegression(optimizer) / LogisticRegression(optimizer)"] --> B[".fit(data)"]
    B --> C["validate_optimizer(optimizer, allowed_optimizers, model_name)"]
    C --> D["_build_model(n_features)<br/>Sequential(DenseLayer(activation, ZeroParametersInitializer))"]
```

## Scenario 1 — closed form

`LinearRegression._solve_closed_form` solves the normal equation directly:

```python
def _solve_closed_form(self, X_augmented, y):
    return np.matmul(
        np.linalg.inv(np.matmul(X_augmented.T, X_augmented)),
        np.matmul(X_augmented.T, y),
    )
```

`X_augmented` is `[X | 1]` — a trailing column of ones folds the intercept
into the same matrix solve, so the vector that comes back is
`[w1, ..., wp, b]`, intercept last. `unpack_flat_weights` (further down) is
what splits that flat vector back into the layer's separate `W` and `b`.

`GLMAbstract` itself only *declares* `_solve_closed_form` — its default
implementation raises, since not every GLM has one in closed form.
`LogisticRegression` never overrides it, and never lists `"closed form"` in
`_allowed_optimizers` either, so the request is rejected twice over: first
by the allow-list check in `fit`, and — for any path that skipped that
check — again by the abstract method itself.

In [2]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet

rng = np.random.default_rng(0)
n = 200
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
y = 2.0 * x1 - 1.0 * x2 + 0.5 + rng.normal(0, 0.3, n)   # true: w = [2, -1], b = 0.5

df = pl.DataFrame({"x1": x1, "x2": x2, "y": y})
data = DolceSet()
data.load_from_polars_dataframe(df, target_col="y")

cf_model = LinearRegression().fit(data)          # optimizer="closed form"

# Reproduce _solve_closed_form by hand, on the same [X | 1]
X_aug = np.column_stack((data.X, np.ones(len(data.X))))
w_by_hand = np.linalg.inv(X_aug.T @ X_aug) @ (X_aug.T @ data.y)

print("model weights/bias:", np.round(cf_model.weights.ravel(), 4), np.round(cf_model.bias, 4))
print("by hand [w1, w2, b]:", np.round(w_by_hand, 4))

try:
    LogisticRegression("closed form").fit(data)
except ValueError as e:
    print("\nLogisticRegression rejects it:", e)

model weights/bias: [ 2.0201 -0.9821] [0.5028]
by hand [w1, w2, b]: [ 2.0201 -0.9821  0.5028]

LogisticRegression rejects it: optimizer 'closed form' is not supported by LogisticRegression. Allowed: ['gradient descent', 'newton'].


## Scenario 2 — gradient descent, through the shared `Trainer`

`_fit_iterative` doesn't write its own loop. It builds a `Trainer` around
the same one-layer `Sequential`, with `GradientDescent(momentum_type="nesterov")`
as the default optimizer and `MiniBatchSampler()` as the default sampler,
then calls `trainer.run(data, ...)`. `Trainer` is the one place
forward → backward → step is sequenced, for GLMs, general networks and the
perceptron alike (see [`05_optimization`](05_optimization.ipynb) for that
loop's own deep dive).

Because a GLM really is just a one-layer network, we can bypass the
`LinearRegression` wrapper entirely and reconstruct the exact same fit by
hand — same layer, same loss, same optimizer, same sampler, wired into a
`Trainer` directly:

In [3]:
from dolcestat.core.activations import LU
from dolcestat.core.losses import MeanSquaredError
from dolcestat.core.parameters import ZeroParametersInitializer
from dolcestat.core.samplers import MiniBatchSampler
from dolcestat.core.trainer import Trainer
from dolcestat.neural_networks.layers import DenseLayer
from dolcestat.neural_networks.networks import Sequential
from dolcestat.optimization.gradient_descent import GradientDescent

np.random.seed(0)   # mini-batch sampler: seed for reproducibility
gd_model = LinearRegression("gradient descent").fit(data)

np.random.seed(0)   # same seed -> the manual Trainer sees the same batches
manual_layer = DenseLayer(
    activation=LU(),
    input_size=2,
    output_size=1,
    parameters_initializer=ZeroParametersInitializer(),
)
manual_net = Sequential(manual_layer)
trainer = Trainer(
    model=manual_net,
    loss=MeanSquaredError(),
    optimizer=GradientDescent(momentum_type="nesterov"),
    sampler=MiniBatchSampler(),
)
manual_history = trainer.run(data, n_epochs=1000, tol=1e-6)

print("LinearRegression wrapper:", np.round(gd_model.weights.ravel(), 4), np.round(gd_model.bias, 4))
print("hand-built Sequential:   ", np.round(manual_layer.W.value.ravel(), 4), np.round(manual_layer.b.value, 4))
print("epochs to converge, wrapper vs. manual:", gd_model.history.n_epochs, manual_history.n_epochs)

LinearRegression wrapper: [ 2.0125 -0.9693] [0.498]
hand-built Sequential:    [ 2.0125 -0.9693] [0.498]
epochs to converge, wrapper vs. manual: 136 136


## Scenario 3 — Newton's method, the Hessian

Newton is deliberately **not** an `Optimizer`. It needs the design matrix
and a single joint Hessian over one flat weight vector — a shape the
`step(parameters)` protocol used by `Trainer`/`GradientDescent` can't
express — so `GLMAbstract` gives it its own loop, `_solve_newton`, right
next to `_solve_closed_form`:

```python
for iteration in range(self._newton_max_iters):
    predictions = self._linear_predictor(X_augmented, w_flat)
    loss_value = loss.forward(predictions, y, training=False)
    ...
    gradient = self._newton_gradient(X_augmented, y, predictions)
    hessian = self._newton_hessian(X_augmented, y, predictions)
    w_flat = w_flat - np.matmul(np.linalg.inv(hessian), gradient)
    ...  # stop once the loss stops moving
```

Every other line is shared; only `_newton_gradient`/`_newton_hessian` are
abstract, implemented once per model:

| | gradient | Hessian |
|---|---|---|
| `LinearRegression` | `(2/n) Xᵀ(p − y)` | `(2/n) XᵀX` — **constant**, doesn't depend on `w` |
| `LogisticRegression` | `(1/n) Xᵀ(p − y)` | `(1/n) Xᵀ diag(p(1−p)) X` — depends on the current `p` |

Because the squared-error loss is exactly quadratic in `w`, its Hessian
never changes — so Newton's *first* step already lands on the exact
minimum, matching the closed form. Logistic regression's loss curves, so
its Hessian is recomputed from the current `p` at every iteration; this is
exactly the classical **IRLS** (iteratively reweighted least squares)
algorithm for fitting a GLM, re-derived here from the same gradient/Hessian
pair rather than assumed.

In [4]:
newton_model = LinearRegression("newton").fit(data)

print("closed form :", np.round(cf_model.weights.ravel(), 4), np.round(cf_model.bias, 4))
print("newton      :", np.round(newton_model.weights.ravel(), 4), np.round(newton_model.bias, 4))
print("per-epoch loss:", [round(l, 5) for l in newton_model.history.epoch_loss])
# the loss already bottoms out after the very first update (epoch 1); the
# remaining epoch(s) are just the tolerance check confirming convergence.

closed form : [ 2.0201 -0.9821] [0.5028]
newton      : [ 2.0201 -0.9821] [0.5028]
per-epoch loss: [5.51242, 0.08927, 0.08927]


Logistic regression has no closed form, but the same Hessian mechanics
apply. Newton needs curvature at every step — each iteration inverts a
p×p matrix — but reaches the optimum in a handful of iterations; gradient
descent needs many cheap steps instead:

In [5]:
prob = 1 / (1 + np.exp(-(1.5 * x1 - 2.0 * x2 + 0.4)))
label = (rng.uniform(size=n) < prob).astype(int)

clf_data = DolceSet()
clf_data.load_from_polars_dataframe(
    pl.DataFrame({"x1": x1, "x2": x2, "label": label}), target_col="label"
)

np.random.seed(0)
clf_gd = LogisticRegression("gradient descent").fit(clf_data)
clf_newton = LogisticRegression("newton").fit(clf_data)

print("gradient descent:", np.round(clf_gd.weights.ravel(), 4), np.round(clf_gd.bias, 4), f"({clf_gd.history.n_epochs} epochs)")
print("newton:          ", np.round(clf_newton.weights.ravel(), 4), np.round(clf_newton.bias, 4), f"({clf_newton.history.n_epochs} epochs)")

gradient descent: [ 1.3018 -1.8048] [0.1095] (287 epochs)
newton:           [ 1.4762 -2.1849] [0.1752] (6 epochs)


## Recap

One abstract class, one shared one-layer network, three interchangeable
ways to find good values for `W` and `b` — an exact solve, the shared
`Trainer` loop, and a hand-rolled Newton loop — all writing into the same
`Parameters`, which is why `predict` never needs to know which one ran.

For the *usage* of these models, see
[`02_linear_regression`](02_linear_regression.ipynb) and
[`03_logistic_regression`](03_logistic_regression.ipynb). For a closer look
at the optimizers and samplers that power the gradient-descent path —
including the momentum types and batching strategies — continue to
[`05_optimization`](05_optimization.ipynb).

```mermaid
flowchart TD
    D["Sequential(DenseLayer(activation, ZeroParametersInitializer))"] --> E{"optimizer"}
    E -->|"'closed form'"| F["_fit_direct(data)"]
    E -->|"'newton'"| F
    E -->|"'gradient descent' or Optimizer instance"| G["_fit_iterative(data)"]
    F --> H{"optimizer"}
    H -->|"'closed form'"| I["_solve_closed_form(X_augmented, y)"]
    H -->|"'newton'"| J["_solve_newton(X_augmented, y)"]
    I --> K["unpack_flat_weights(w_flat, layer)"]
    J --> K
    G --> L["Trainer(model, loss, optimizer, sampler).run(data, n_epochs, tol)"]
    K --> M["is_fitted = True"]
    L --> M
    M --> N["training_metrics = predict(data)"]

    classDef closedForm fill:#a5d8ff,stroke:#1971c2,color:#0b3d63
    classDef gradientDescent fill:#ffe066,stroke:#e8990c,color:#5c4500
    classDef newton fill:#ffa8a8,stroke:#c92a2a,color:#5c0d0d

    class I,K closedForm
    class L gradientDescent
    class J newton
```